# JSC Translator - full training run

Fine-tunes NLLB-200-distilled-600M on all 29 `data_ready` Cameroonian languages (see `data/languages.json`) using LoRA.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

**One-time setup - GitHub token (repo is private):**
1. Create a token at https://github.com/settings/tokens with `repo` scope.
2. In this notebook, click the key icon (Secrets) in the left sidebar.
3. Add a secret named `GITHUB_TOKEN` with that token as the value, and enable notebook access.

**If this session disconnects partway through training:** just reconnect (`Runtime > Reconnect`) and re-run all cells from the top (`Runtime > Run all`). Checkpoints are saved to Google Drive every `--eval-steps` steps, and the training cell auto-resumes from the latest one - it does not start over.

**On runtime:** this is a big run (1.69M rows, 29 languages). The first training cell logs its steps/sec after ~50 steps - check that early and extrapolate before assuming it'll finish in one session. It very likely won't; multiple reconnect-and-resume cycles are expected and fine.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Checkpoints and the final merged model live in Drive, not the ephemeral
# Colab VM disk, so they survive a disconnect.
OUT_DIR = '/content/drive/MyDrive/jsc_translator/model_out'
import os
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
assert GITHUB_TOKEN, (
    "GITHUB_TOKEN secret is empty or not granted to this notebook - "
    "click the key icon in the left sidebar, add it, and enable notebook access."
)
REPO = 'Steco17/jsc-api-backend'
REPO_DIR = '/content/jsc_api_backend'

import os, shutil

# A directory can exist here from a previous failed clone attempt (e.g. bad
# token) without actually containing the repo - checking for a file we know
# is in it, not just directory existence, catches that case and re-clones.
if not os.path.isfile(f'{REPO_DIR}/requirements-train.txt'):
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR}

assert os.path.isfile(f'{REPO_DIR}/requirements-train.txt'), (
    "Clone failed - scroll up to the git clone output above for the actual "
    "error (commonly: bad/expired token, or the token's repo access doesn't "
    "include jsc-api-backend)."
)
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements-train.txt

# Colab preinstalls an old torchao that peft's LoRA setup probes (as one of
# several optional quantization backends) and errors on, even though this
# project never uses torchao. Not needed here, so remove it rather than
# fight pip's dependency resolver over a version we don't care about.
!pip uninstall -qy torchao

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU - set Runtime > Change runtime type > T4 GPU"

In [ ]:
# Regenerate the combined dataset from the raw CSVs (not committed to git -
# it's fully derivable and would have bloated the repo).
!python scripts/prepare_all.py data -o data/prepared

In [ ]:
import json

EXPECTED_LANGS = {
    'agq_Latn', 'ags_Latn', 'azo_Latn', 'bbj_Latn', 'bbk_Latn', 'bfd_Latn',
    'bmo_Latn', 'bri_Latn', 'bsq_Latn', 'bum_Latn', 'bwt_Latn', 'etu_Latn',
    'ewo_Latn', 'fub_Latn', 'gya_Latn', 'ksf_Latn', 'lem_Latn', 'lmp_Latn',
    'lns_Latn', 'mcu_Latn', 'mgo_Latn', 'muy_Latn', 'nge_Latn', 'oku_Latn',
    'pny_Latn', 'tui_Latn', 'vut_Latn', 'xmg_Latn', 'yat_Latn',
}

found = set()
with open('data/prepared/train.jsonl', encoding='utf-8') as f:
    for line in f:
        found.add(json.loads(line)['tgt_lang'])

missing = EXPECTED_LANGS - found
assert not missing, (
    f"data/prepared/train.jsonl is missing {len(missing)} language(s): "
    f"{sorted(missing)}. The prepare_all.py cell above likely got "
    "interrupted before finishing - re-run it to completion (don't stop it "
    "partway through) before training."
)
print(f"All {len(EXPECTED_LANGS)} languages present in train.jsonl.")

In [ ]:
NEW_LANGS = (
    'agq_Latn ags_Latn azo_Latn bbj_Latn bbk_Latn bfd_Latn bmo_Latn bri_Latn '
    'bsq_Latn bum_Latn bwt_Latn etu_Latn ewo_Latn fub_Latn gya_Latn ksf_Latn '
    'lem_Latn lmp_Latn lns_Latn mcu_Latn mgo_Latn muy_Latn nge_Latn oku_Latn '
    'pny_Latn tui_Latn vut_Latn xmg_Latn yat_Latn'
)

# batch/grad-accum tuned for a T4's 16GB (local smoke testing used batch=2
# on a 4GB card). Watch the first ~50 steps' it/s in the output below, then
# extrapolate total runtime - adjust --epochs down if it won't fit your
# available session time.
!python scripts/finetune.py \
  --train data/prepared/train.jsonl \
  --dev data/prepared/dev.jsonl \
  --new-langs {NEW_LANGS} \
  --out {OUT_DIR} \
  --epochs 4 --batch 16 --grad-accum 4 --eval-steps 500

## After training finishes

`<OUT_DIR>/merged/` (the `OUT_DIR` set two cells above, under `/content/drive/MyDrive/jsc_translator/model_out`) now holds the full fine-tuned model - already in Drive, already safe. Convert it to CTranslate2 int8 for CPU serving:

In [ ]:
CT2_DIR = '/content/drive/MyDrive/jsc_translator/model_ct2'
!ct2-transformers-converter --model {OUT_DIR}/merged --output_dir {CT2_DIR} --quantization int8

Download `model_out/merged/` and `model_ct2/` from Drive to run `app/main.py` locally (`MODEL_DIR`/`TOKENIZER_DIR` env vars), or run `scripts/evaluate.py` against `data/prepared/test.jsonl` first to check quality per language pair before deploying.

Once you're happy with it, flip each trained language's status from `data_ready` to `fine_tuned` in `data/languages.json` and commit - that's what makes `app/main.py` actually expose them via `/translate`.